# 65 - Anisotropic Adaptive Kalman Measurement Covariance

This notebook validates the small code change that lets the UltraTimTrack Python Kalman path use separate adaptive measurement noise for orientation and length-side geometry.

Previous behavior:

`R_t = R_base * r_scale(combined_confidence)`

New optional behavior:

`R_t = diag([R_theta_0 * s_theta, R_L_0 * s_L])`

The MATLAB-compatible two-state implementation stores its internal diagonal in state order `[x_sup, alpha]`, so the reported `measurement_R_diag` is `[R_L, R_theta]` while the conceptual covariance is `[theta, length]`.

## 1. Imports and API Surface

In [1]:
from __future__ import annotations

import inspect
import subprocess
import sys
from pathlib import Path

import numpy as np

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from ultrasound_tracker.speckle_confidence import (
    SpeckleConfidenceConfig,
    adapt_anisotropic_measurement_covariance,
    anisotropic_confidence_to_r_scales,
    combine_anisotropic_confidence_metrics,
    confidence_to_r_scale,
)
from ultrasound_tracker.ultratimtrack_matlab_2state import (
    MatlabTwoStateKalmanConfig,
    run_matlab_2state_kalman,
)

print('combine_anisotropic_confidence_metrics:', inspect.signature(combine_anisotropic_confidence_metrics))
print('anisotropic_confidence_to_r_scales:', inspect.signature(anisotropic_confidence_to_r_scales))
print('run_matlab_2state_kalman:', inspect.signature(run_matlab_2state_kalman))

combine_anisotropic_confidence_metrics: (metrics: 'ConfidenceMetrics | Mapping[str, Any]', theta_weights: 'Optional[Mapping[str, float]]' = None, length_weights: 'Optional[Mapping[str, float]]' = None, config: 'Optional[SpeckleConfidenceConfig]' = None) -> 'dict[str, float]'
anisotropic_confidence_to_r_scales: (confidence_theta: 'float', confidence_length: 'float', config: 'Optional[SpeckleConfidenceConfig]' = None) -> 'dict[str, float]'
run_matlab_2state_kalman: (klt_segments: 'np.ndarray', timtrack_alpha_deg: 'np.ndarray', superficial_apo_lines: 'np.ndarray', deep_apo_lines: 'np.ndarray', *, config: 'MatlabTwoStateKalmanConfig | None' = None, fixed_superficial_y: 'Optional[float]' = None, mm_per_pixel: 'Optional[float]' = None, measurement_r_scale: 'Optional[np.ndarray]' = None, measurement_r_scale_theta: 'Optional[np.ndarray]' = None, measurement_r_scale_length: 'Optional[np.ndarray]' = None) -> 'Dict[str, np.ndarray]'


## 2. Confidence Split Examples

The theta confidence is weighted toward feature reliability, speckle coherence, motion coherence, and orientation stability. The length confidence is weighted toward length plausibility, geometry stability, aponeurosis stability, and intersection quality when those are available.

In [2]:
cfg = SpeckleConfidenceConfig(r_min_scale=1.0, r_max_scale=20.0, r_gamma=1.5)

low_theta_metrics = {
    'speckle_confidence': 0.90,
    'motion_consistency': 0.90,
    'feature_reliability': 0.15,
    'geometry_alpha_score': 0.20,
    'geometry_angle_jump_score': 0.20,
    'geometry_stability': 0.95,
    'geometry_length_score': 0.95,
    'geometry_length_jump_score': 0.95,
    'aponeurosis_stability': 0.95,
    'intersection_quality': 0.95,
}
low_length_metrics = {
    'speckle_confidence': 0.95,
    'motion_consistency': 0.95,
    'feature_reliability': 0.95,
    'geometry_alpha_score': 0.95,
    'geometry_angle_jump_score': 0.95,
    'geometry_stability': 0.20,
    'geometry_length_score': 0.15,
    'geometry_length_jump_score': 0.15,
    'aponeurosis_stability': 0.15,
    'intersection_quality': 0.15,
}

for label, metrics in [('low theta', low_theta_metrics), ('low length', low_length_metrics)]:
    conf = combine_anisotropic_confidence_metrics(metrics, config=cfg)
    scales = anisotropic_confidence_to_r_scales(conf['confidence_theta'], conf['confidence_length'], cfg)
    R_theta_length = adapt_anisotropic_measurement_covariance(
        R_theta_base=3.0,
        R_length_base=100.0,
        confidence_theta=conf['confidence_theta'],
        confidence_length=conf['confidence_length'],
        config=cfg,
    )
    print(label)
    print('  confidence:', conf)
    print('  scales:', scales)
    print('  diag([R_theta, R_L]):', np.diag(R_theta_length))

low theta
  confidence: {'confidence_theta': 0.3558336939010577, 'confidence_length': 0.95}
  scales: {'r_scale_theta': 10.823146207367646, 'r_scale_length': 1.2124264578624804}
  diag([R_theta, R_L]): [ 32.46943862 121.24264579]
low length
  confidence: {'confidence_theta': 0.95, 'confidence_length': 0.15661453650702545}
  scales: {'r_scale_theta': 1.2124264578624804, 'r_scale_length': 15.716101272815182}
  diag([R_theta, R_L]): [   3.63727937 1571.61012728]


## 3. Kalman Diagonal R Behavior

The two-state filter still updates scalar states independently. The new arguments simply choose different per-frame measurement variances for the x/length side and alpha/theta side.

In [3]:
klt = np.array(
    [
        [80.0, 10.0, 30.0, 60.0],
        [81.0, 10.0, 31.0, 60.0],
    ],
    dtype=float,
)
superficial = np.tile(np.array([[1.0, 10.0, 101.0, 10.0]]), (2, 1))
deep = np.tile(np.array([[1.0, 60.0, 101.0, 60.0]]), (2, 1))
alpha = np.array([45.0, 44.0])
kalman_cfg = MatlabTwoStateKalmanConfig(
    x_measurement_variance=100.0,
    alpha_measurement_variance=3.0,
    run_smoother=False,
    use_adaptive_R=True,
)

angle_low = run_matlab_2state_kalman(
    klt,
    alpha,
    superficial,
    deep,
    config=kalman_cfg,
    measurement_r_scale_theta=np.array([1.0, 5.0]),
    measurement_r_scale_length=np.array([1.0, 1.0]),
)
length_low = run_matlab_2state_kalman(
    klt,
    alpha,
    superficial,
    deep,
    config=kalman_cfg,
    measurement_r_scale_theta=np.array([1.0, 1.0]),
    measurement_r_scale_length=np.array([1.0, 5.0]),
)
global_old = run_matlab_2state_kalman(
    klt,
    alpha,
    superficial,
    deep,
    config=kalman_cfg,
    measurement_r_scale=np.array([1.0, 5.0]),
)

print('Internal order is [R_L/x, R_theta/alpha]')
print('low theta only:', angle_low['measurement_R_diag'][1])
print('low length only:', length_low['measurement_R_diag'][1])
print('old global fallback:', global_old['measurement_R_diag'][1])

Internal order is [R_L/x, R_theta/alpha]
low theta only: [100.  15.]
low length only: [500.   3.]
old global fallback: [500.  15.]


## 4. Focused Tests

In [4]:
cmd = [
    sys.executable,
    '-m',
    'pytest',
    'tests/test_speckle_confidence.py',
    'tests/test_ultratimtrack_matlab_2state.py',
]
print(' '.join(cmd))
completed = subprocess.run(cmd, cwd=ROOT, text=True, capture_output=True)
print(completed.stdout)
if completed.stderr:
    print(completed.stderr)
assert completed.returncode == 0

/Users/grosbedou/PycharmProjects/NDORMS/.venv/bin/python -m pytest tests/test_speckle_confidence.py tests/test_ultratimtrack_matlab_2state.py


============================= test session starts ==============================
platform darwin -- Python 3.11.9, pytest-9.1.0, pluggy-1.6.0
rootdir: /Users/grosbedou/PycharmProjects/NDORMS
plugins: anyio-4.13.0
collected 15 items

tests/test_speckle_confidence.py ............                            [ 80%]
tests/test_ultratimtrack_matlab_2state.py ...                            [100%]

============================== 15 passed in 0.59s ==============================



## 5. Implementation Notes

- `combine_confidence_metrics` and `measurement_r_scale` are preserved for backward compatibility.
- New callers can use `confidence_theta -> r_scale_theta` and `confidence_length -> r_scale_length`.
- If only `measurement_r_scale` is supplied, both diagonal entries are scaled exactly as before.
- In the current MATLAB-compatible state, length is not an explicit state; the length-side adaptive variance scales the superficial attachment `x` measurement, which is the state most directly controlling the reconstructed fascicle intersection and length.